<a href="https://colab.research.google.com/github/SafaaMahbub/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order','vendor_id'])
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum()==7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendor_df = pd.json_normalize(vendors_json)
joined = lines.merge(vendor_df, on='vendor_id', how='left', indicator=True)
joined

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


Vendor 07 is unmatched and 3 units are attached to it

Since we did a left join on the df table, that means that everything from the left table will be merged with the right even if there is not a matching vendor_id in the right table. Hence in the order df there is a order 3 which vnedor is vendor-07 but that vendor is not listed in the vendor list. Because of that, the zone nor the na,e could be retrieved in the join.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
total_qty = joined.groupby("name",dropna=False)["qty"].sum()
total_qty

,qty
name,
Cav Merch,1
Hoos Burgers,3
NaN,3


The unmatched vendor appears as NaN since it is a legit vendor whose imformation has to be investigated more into to retrieve the name and other information. I did not drop the na because the quantity number is significant

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
zone_df = joined.groupby("zone", dropna=False)['qty'].sum()
zone_df

,qty
zone,
A,4
NaN,3


Zone A sold the most items with 4 units in total. The zone total is 7 but 3 of them are listed under the uknown vendor vendor-07 which did not match the vendor list and therefore had no zone.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [7]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
for ragged_order in ragged_json:
  if 'lines' not in ragged_order:
    ragged_order['lines']=[]

corrected_df = pd.json_normalize(ragged_json,record_path='lines',meta=['order','vendor_id'])

print('order 5 has no lines, so it does not appear in the dataFrame')
corrected_df

#adding a column that reports quantities
corrected_df['qty_known'] = corrected_df['qty'].notna()
corrected_df


order 5 has no lines, so it does not appear in the dataFrame


,item,qty,order,vendor_id,qty_known
0,Soda,2.0,4,V-01,True
1,Fries,NaN,6,V-01,False


### Q6 — Validate

In [8]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** one row per line is useful because each item can be analysed seperately. It can be used to look at individual quantities for an item seperatley. Questions like "How many units were sold for each dish".

**b)** For a "NaN" missign quantity, we do not know how many were actually sold and it can indicate that we ned to investigate this particular row. A row mit 0 quantity essentially mean that for that particular item, no units were sold. If we decided to just treat/transform NaN to 0, we would reflect incorrect calculations such as revenue. For example, the total unit sold would be lower than the actual total because we are "assuming" that NaN means 0 which may not be the case